# 概率、校准与回退

这是本站的**合成数据教学实验**，不调用 Jev 或任何模型。无需密钥、安装包或 GPU。

任务：先运行，再修改一个高置信度错误样本，保存观察和解释。

In [ ]:
labels = [1, 0, 1, 0, 1, 0, 1, 0]
probabilities = [0.95, 0.10, 0.80, 0.85, 0.60, 0.40, 0.55, 0.20]
assert len(labels) == len(probabilities)

def brier(labels, probabilities):
    return sum((p-y)**2 for y, p in zip(labels, probabilities))/len(labels)

def selective(labels, probabilities, threshold):
    # 此处使用二元预测的最大类别概率；不是 TypeSafe confidence 公式。
    kept = [(y, p) for y,p in zip(labels, probabilities) if max(p,1-p) >= threshold]
    errors = sum((p >= 0.5) != bool(y) for y,p in kept)
    return {"threshold":threshold, "coverage":len(kept)/len(labels), "accepted_error":errors/len(kept) if kept else None}

print("Brier:", round(brier(labels, probabilities), 4))
for threshold in [0.6,0.9]:
    print(selective(labels, probabilities, threshold))

## 只改变一个变量

下面把第一个原本正确的高置信度样本标签翻转。两次使用相同预测，观察 Brier 和自动处理子集的错误率如何变化。小样本没有统计结论。

In [ ]:
changed = labels.copy()
changed[0] = 0
print("Changed Brier:", round(brier(changed, probabilities), 4))
print(selective(changed, probabilities, 0.9))
assert brier(changed, probabilities) > brier(labels, probabilities)
assert selective(changed, probabilities, 0.9)["accepted_error"] > 0

## 写下你的理解

1. 高概率为什么仍会答错？
2. 阈值升高时覆盖率发生了什么？
3. 如果低置信度请求交给另一个模型，如何计算整个系统成本与时延？
4. 真实评价需要验证集选阈值、独立测试集验收；不要用这八个样本评价 Jev。

回到 [学习单元](https://suguilin.github.io/study/?id=digest-jev#study-practice)，在第三题保存修改与输出。

原始来源：[TypeSafe confidence](https://docs.typesafe.ai/confidence)、[独立评测摘要](https://arxiv.org/abs/2609.37647)。